## H3 Anime Studio：畫質 AI 升級、中文配音、繁體字幕
這是 H3 影片的後製 Notebook：從 Google Drive 選取最新 MP4，逐幀用 Real-ESRGAN 動漫模型 AI 增強，透過 Qwen3-TTS 生成中文配音、Whisper 辨識並轉繁體字幕，最後用 FFmpeg 合成。

請在 **新的 Colab GPU 執行階段** 執行。無須開啟 8188 網頁，僅使用本機 ComfyUI API，可避開先前的 403。先用 5 秒測試再改完整影片。H3 本身請用原本的筆記本生成並保存到 Google Drive。未在你的 Colab 實際端到端測試。

In [ ]:
# ① 設定（先從這裡修改）
INPUT_VIDEO = ""  # 留空：自動讀取 Drive 裡最新的 H3 MP4。也可貼入檔案完整路徑
OUTPUT_W, OUTPUT_H = 720, 1280   # L4 建議先720p；A100可試1080x1920
AI_UPSCALE = True                # True: 真正的 Real-ESRGAN AI 放大；False: 快速 FFmpeg 放大
TEST_SECONDS = 5                 # 首次5秒測試；正式全片設 0
VOICE_TEXT = "吾以天命之名，斬破這無盡的黑暗！"
VOICE_SPEAKER = "Uncle_Fu"       # Vivian=女聲；Serena=柔和女聲；Uncle_Fu=低沉男聲
VOICE_STYLE = "用有力、充滿決心的語氣說，語速偏慢。"
USE_QWEN_TTS = True
WHISPER_SUBTITLES = True
BURN_SUBTITLES = True
BACKGROUND_VOLUME = 0.20

import sys, os, json, shutil, subprocess, time, uuid, gc
from pathlib import Path
from fractions import Fraction
from IPython.display import display,Video
from google.colab import drive
import torch

def sh(args,cwd=None):
    print("Running",str(args)[:180])
    subprocess.run([str(x) for x in args],cwd=str(cwd) if cwd else None,check=True)
def info(path):
    return json.loads(subprocess.check_output(["ffprobe","-v","quiet","-show_streams","-show_format","-of","json",str(path)],text=True))
assert torch.cuda.is_available(),"請先在 Colab 選 GPU！"
print("GPU：",torch.cuda.get_device_name(0))
drive.mount("/content/drive")
D=Path("/content/drive/MyDrive")
OUT=D/"MiniMax_H3_Studio_Final"
OUT.mkdir(exist_ok=True)
W=Path("/content/H3_Studio_Final")
W.mkdir(exist_ok=True)
if INPUT_VIDEO.strip():
    source=Path(INPUT_VIDEO)
else:
    options=[]
    for folder in [D/"MiniMax_H3_Outputs",D/"H3_Videos",D/"H3_Anime_Studio"]:
        if folder.exists():options.extend(folder.rglob("*.mp4"))
    if not options:
        from google.colab import files
        print("找不到 Drive 影片，請手動上傳：")
        uploaded=files.upload()
        if not uploaded: raise ValueError("你沒有上傳影片")
        source=Path("/content")/next(iter(uploaded.keys()))
    else:
        source=max(options,key=lambda x:x.stat().st_mtime)
if not source.exists():raise FileNotFoundError(source)
src=W/"source.mp4"
if source.resolve()!=src.resolve():shutil.copy2(source,src)
metadata=info(src)
v=next(s for s in metadata["streams"] if s["codec_type"]=="video")
HAS_AUDIO=any(s["codec_type"]=="audio" for s in metadata["streams"])
fps=float(Fraction(v.get("avg_frame_rate","24/1")))
duration=float(metadata["format"].get("duration",0))
clip_len=min(duration,TEST_SECONDS) if TEST_SECONDS else duration
print("找到影片：",source,"畫面：",v["width"],v["height"],"fps：",fps,"處理秒數：",clip_len)


In [ ]:
# ② 抽取影格，使用 Real-ESRGAN Anime 6B 逐格提升畫質
import urllib.request, requests
sh(["apt-get","update","-qq"])
sh(["apt-get","install","-y","-qq","ffmpeg","fonts-noto-cjk"])
raw=W/"frames_source"
enhanced=W/"frames_ai"
for folder in [raw,enhanced]:
    folder.mkdir(exist_ok=True)
    for f in folder.glob("*.png"): f.unlink()
cmd=["ffmpeg","-hide_banner","-loglevel","error","-y","-i",src]
if TEST_SECONDS:cmd+=["-t",str(clip_len)]
cmd+=["-vsync","0",raw/"%06d.png"]
sh(cmd)
parts=sorted(raw.glob("*.png"))
if not parts:raise ValueError("沒有影格")
print("影格數",len(parts))

if AI_UPSCALE:
    C=Path("/content/ComfyUI")
    if not (C/"main.py").exists():
        sh(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",C])
    sh([sys.executable,"-m","pip","install","-q","-r",C/"requirements.txt"])
    model=C/"models/upscale_models/RealESRGAN_x4plus_anime_6B.pth"
    model.parent.mkdir(parents=True,exist_ok=True)
    if not model.exists():
        urllib.request.urlretrieve("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",model)
    if model.stat().st_size<1000000:raise RuntimeError("Real-ESRGAN 權重不完整")
    (C/"input").mkdir(exist_ok=True)
    (C/"output").mkdir(exist_ok=True)
    url="http://127.0.0.1:8188"
    def ready():
        try:return requests.get(url+"/system_stats",timeout=2).status_code==200
        except Exception:return False
    if not ready():
        log=open("/content/H3StudioComfy.log","a")
        SERVER=subprocess.Popen([sys.executable,"main.py","--listen","127.0.0.1","--port","8188","--disable-pinned-memory"],cwd=C,stdout=log,stderr=subprocess.STDOUT)
        for _ in range(150):
            if ready():break
            if SERVER.poll() is not None:raise RuntimeError("ComfyUI 啟動失敗，請查看 /content/H3StudioComfy.log")
            time.sleep(2)
    if not ready():raise RuntimeError("ComfyUI 無法啟動")
    for idx,srcpng in enumerate(parts):
        file="studio_"+srcpng.name
        shutil.copy2(srcpng,C/"input"/file)
        graph={
          "1":{"class_type":"LoadImage","inputs":{"image":file}},
          "2":{"class_type":"UpscaleModelLoader","inputs":{"model_name":model.name}},
          "3":{"class_type":"ImageUpscaleWithModel","inputs":{"upscale_model":["2",0],"image":["1",0]}},
          "4":{"class_type":"ImageScale","inputs":{"image":["3",0],"upscale_method":"lanczos","width":OUTPUT_W,"height":OUTPUT_H,"crop":"center"}},
          "5":{"class_type":"SaveImage","inputs":{"images":["4",0],"filename_prefix":"H3studio/upscale"}}
        }
        r=requests.post(url+"/prompt",json={"prompt":graph},timeout=30)
        if r.status_code!=200:raise RuntimeError("ComfyUI 節點驗證錯誤："+r.text[:2500])
        pid=r.json()["prompt_id"]
        result=None
        for _ in range(240):
            time.sleep(2)
            hist=requests.get(url+"/history/"+pid,timeout=30).json()
            if pid in hist:
                result=hist[pid]
                if result.get("status",{}).get("status_str")=="error":
                    raise RuntimeError("Real-ESRGAN 無法生成："+str(result["status"])[:1500])
                if result.get("status",{}).get("completed"):break
        if not result or not result.get("status",{}).get("completed"):
            raise TimeoutError("處理影格逾時："+file)
        items=result["outputs"]["5"]["images"]
        f=items[0]
        shutil.copy2(C/"output"/f.get("subfolder","")/f["filename"],enhanced/srcpng.name)
        if (idx+1)%10==0 or idx+1==len(parts):print("AI 放大進度：",idx+1,"/",len(parts))
    # 釋放顯存，準備 Qwen3-TTS
    try:requests.post(url+"/free",json={"unload_models":True,"free_memory":True},timeout=20)
    except Exception:pass
    if "SERVER" in globals() and SERVER.poll() is None:SERVER.terminate()
else:
    from PIL import Image,ImageFilter
    for png in parts:
        with Image.open(png) as im:
            im=im.convert("RGB").resize((OUTPUT_W,OUTPUT_H),Image.Resampling.LANCZOS)
            im=im.filter(ImageFilter.UnsharpMask(radius=1.0,percent=100,threshold=2))
            im.save(enhanced/png.name)

upscaled=W/"upscaled.mp4"
cmd=["ffmpeg","-hide_banner","-loglevel","error","-y","-framerate",str(fps),"-i",enhanced/"%06d.png","-i",src,"-map","0:v:0"]
if HAS_AUDIO:cmd+=["-map","1:a:0"]
cmd+=["-c:v","libx264","-crf","17","-pix_fmt","yuv420p"]
if HAS_AUDIO:cmd+=["-c:a","aac"]
cmd+=["-t",str(clip_len),upscaled]
sh(cmd)
print("✅ 已輸出畫質增強影片：",upscaled)


In [ ]:
# ③ Qwen3-TTS 自然中文配音（官方內建聲線）
voice=W/"voice_zh.wav"
if USE_QWEN_TTS and VOICE_TEXT.strip():
    sh([sys.executable,"-m","pip","install","-q","-U","qwen-tts","soundfile"])
    import soundfile as sf
    from qwen_tts import Qwen3TTSModel
    tts=Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
        device_map="cuda:0",dtype=torch.bfloat16,attn_implementation="sdpa")
    wavs,sr=tts.generate_custom_voice(text=VOICE_TEXT,language="Chinese",speaker=VOICE_SPEAKER,instruct=VOICE_STYLE)
    sf.write(str(voice),wavs[0],sr)
    del tts
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ 生成中文配音：",voice)
else:print("略過語音生成")


In [ ]:
# ④ 自動生成繁體字幕檔 SRT（Whisper辨識）
srt=W/"voice_zh_tw.srt"
def stamp(sec):
    milli=max(0,round(sec*1000))
    h,r=divmod(milli,3600000);m,r=divmod(r,60000);s,ms=divmod(r,1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"
if WHISPER_SUBTITLES and voice.exists():
    sh([sys.executable,"-m","pip","install","-q","faster-whisper","opencc-python-reimplemented"])
    from opencc import OpenCC
    from faster_whisper import WhisperModel
    cn=OpenCC("s2twp")
    asr=WhisperModel("small",device="cpu",compute_type="int8")
    segments,_=asr.transcribe(str(voice),language="zh",beam_size=3)
    lines=[]
    for x in segments:
        if x.start>=clip_len:break
        start,end=max(0,x.start),min(x.end,clip_len)
        if end>start:lines.append((start,end,cn.convert(x.text.strip())))
    if not lines:lines=[(0,clip_len,cn.convert(VOICE_TEXT))]
    textout=""
    for idx,(start,end,t) in enumerate(lines,1):
        textout+=f"{idx}\n{stamp(start)} --> {stamp(end)}\n{t}\n\n"
    srt.write_text(textout,encoding="utf-8")
    print("✅ 中文字幕：",srt)
else:print("略過字幕")


In [ ]:
# ⑤ 最後合成 H3 原聲、中文配音、字幕 → MP4
finished=W/"H3_Anime_Studio_Final.mp4"
voice_ok=USE_QWEN_TTS and voice.exists()
inputs=["-i",str(upscaled)]
if voice_ok:inputs+=["-i",str(voice)]
maps=["-map","0:v:0"]
filters=[]
if voice_ok and HAS_AUDIO:
    filters+=["-filter_complex",f"[0:a:0]volume={BACKGROUND_VOLUME}[b];[1:a:0]volume=1.25[v];[b][v]amix=inputs=2:duration=first:dropout_transition=0[a]"]
    maps+=["-map","[a]"]
elif voice_ok:maps+=["-map","1:a:0"]
elif HAS_AUDIO:maps+=["-map","0:a:0"]
if BURN_SUBTITLES and srt.exists():
    filters+=["-vf",f"subtitles={srt}:force_style='FontName=Noto Sans CJK TC,Fontsize=18,Outline=2,Alignment=2,MarginV=55'"]
    vcodec=["-c:v","libx264","-crf","17","-pix_fmt","yuv420p"]
else:vcodec=["-c:v","copy"]
cmd=["ffmpeg","-hide_banner","-loglevel","error","-y"]+inputs+filters+maps+vcodec
if HAS_AUDIO or voice_ok:cmd+=["-c:a","aac","-b:a","192k"]
cmd+=["-t",str(clip_len),"-movflags","+faststart",finished]
sh(cmd)
shutil.copy2(finished,OUT/finished.name)
if srt.exists():shutil.copy2(srt,OUT/srt.name)
print("🎉 完成，Google Drive：",OUT/finished.name)
print("檔案大小：",round(finished.stat().st_size/1024**2,1),"MiB")
display(Video(str(finished),embed=True,width=480))


## 擴充工作流
進階時序畫質：可改用 ComfyUI 官方 SeedVR2 3B INT8 影片升級範本，取代 Real-ESRGAN。
若要補幀至 48fps，可再安裝 ComfyUI-Frame-Interpolation 的 RIFE 節點；補幀增加的是時間取樣，不會憑空修復錯誤動作。
如果 H3 已經說出台詞，請在 H3 prompt 指明 ambient sound effects only, no dialogue，以免和 Qwen3-TTS 中文台詞重疊。配音必須使用自己的素材或合法授權的聲音參考。
參考來源：Real-ESRGAN https://github.com/xinntao/Real-ESRGAN 、Qwen3-TTS https://github.com/QwenLM/Qwen3-TTS 、SeedVR2 https://docs.comfy.org/tutorials/utility/seedvr2 、VFI https://github.com/Fannovel16/ComfyUI-Frame-Interpolation